# KSI Toronto — Step 2: Data Cleaning, Preprocessing & Feature Engineering/

## Objective
The purpose of this step is to clean and transform the KSI dataset before model building.  
This includes removing irrelevant observations, handling missing values, engineering useful features, encoding categorical variables, scaling numerical features, and preparing the train/test datasets.

## Why this step is important
Raw datasets usually contain missing values, inconsistent formats, and columns that are not useful for prediction.  
If we use the data as-is, the machine learning models may perform poorly or produce misleading results.  
For this reason, data cleaning and transformation are necessary before training classification models.

In [1]:
# ============================================================
# KSI Toronto — Step 2: Data Cleaning & Transformation
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

# Paths
DATA_PATH    = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\data\TOTAL_KSI_3737821728629277523.csv'
OUTPUTS_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\outputs\\'
POWERBI_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\powerbi\\'

os.makedirs(OUTPUTS_PATH, exist_ok=True)
os.makedirs(POWERBI_PATH, exist_ok=True)

# Load raw data
df = pd.read_csv(DATA_PATH)
print('✅ Libraries imported and dataset loaded successfully')
print(f'📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

✅ Libraries imported and dataset loaded successfully
📊 Shape: 18,957 rows × 54 columns


In [2]:
# ============================================================
# 2.1 — Remove Irrelevant Rows
# ============================================================
# Remove 'Property Damage Only' — not relevant for our classification
print(f'Before filtering: {df.shape[0]:,} rows')

df = df[df['ACCLASS'].isin(['Fatal', 'Non-Fatal Injury'])].copy()

print(f'After filtering: {df.shape[0]:,} rows')
print(f'Removed: {18957 - df.shape[0]} rows (Property Damage Only)')

Before filtering: 18,957 rows
After filtering: 18,938 rows
Removed: 19 rows (Property Damage Only)


In [3]:
# ============================================================
# 2.2 — Drop Columns with Too Many Missing Values (> 50%)
# ============================================================
threshold = 50.0
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f'📊 Columns with more than 50% missing values: {len(cols_to_drop)}')
print('='*50)
for col in cols_to_drop:
    print(f'   ❌ {col}: {missing_pct[col]}% missing')

df.drop(columns=cols_to_drop, inplace=True)
print(f'\n✅ Columns dropped: {len(cols_to_drop)}')
print(f'📊 New shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

📊 Columns with more than 50% missing values: 19
   ❌ OFFSET: 79.92% missing
   ❌ FATAL_NO: 95.41% missing
   ❌ PEDTYPE: 82.97% missing
   ❌ PEDACT: 82.98% missing
   ❌ PEDCOND: 82.88% missing
   ❌ CYCLISTYPE: 95.75% missing
   ❌ CYCACT: 95.77% missing
   ❌ CYCCOND: 95.78% missing
   ❌ PEDESTRIAN: 59.45% missing
   ❌ CYCLIST: 89.51% missing
   ❌ MOTORCYCLE: 91.11% missing
   ❌ TRUCK: 93.83% missing
   ❌ TRSN_CITY_VEH: 93.96% missing
   ❌ EMERG_VEH: 99.74% missing
   ❌ PASSENGER: 62.1% missing
   ❌ SPEEDING: 85.79% missing
   ❌ REDLIGHT: 91.67% missing
   ❌ ALCOHOL: 95.73% missing
   ❌ DISABILITY: 97.4% missing

✅ Columns dropped: 19
📊 New shape: 18,938 rows × 35 columns


In [4]:
# ============================================================
# 2.3 — Drop Irrelevant Columns
# ============================================================
# These columns are not useful for prediction
irrelevant_cols = ['OBJECTID', 'INDEX', 'ACCNUM', 'STREET1', 'STREET2',
                   'LATITUDE', 'LONGITUDE', 'x', 'y', 
                   'HOOD_140', 'NEIGHBOURHOOD_140']

df.drop(columns=irrelevant_cols, inplace=True)

print(f'✅ Irrelevant columns dropped: {len(irrelevant_cols)}')
print(f'📊 New shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nRemaining columns:')
for col in df.columns:
    print(f'   • {col}')

✅ Irrelevant columns dropped: 11
📊 New shape: 18,938 rows × 24 columns

Remaining columns:
   • DATE
   • TIME
   • ROAD_CLASS
   • DISTRICT
   • ACCLOC
   • TRAFFCTL
   • VISIBILITY
   • LIGHT
   • RDSFCOND
   • ACCLASS
   • IMPACTYPE
   • INVTYPE
   • INVAGE
   • INJURY
   • INITDIR
   • VEHTYPE
   • MANOEUVER
   • DRIVACT
   • DRIVCOND
   • AUTOMOBILE
   • AG_DRIV
   • HOOD_158
   • NEIGHBOURHOOD_158
   • DIVISION


In [5]:
# ============================================================
# 2.4 — Handle Remaining Missing Values
# ============================================================
print('Missing values before cleaning:')
print('='*45)
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

# Fill categorical columns with 'Unknown'
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('Unknown', inplace=True)

print(f'\n✅ Missing values filled with "Unknown"')
print(f'\nMissing values after cleaning:')
print(df.isnull().sum().sum(), 'total missing values remaining')

Missing values before cleaning:
ROAD_CLASS     486
DISTRICT       229
ACCLOC        5456
TRAFFCTL        75
VISIBILITY      24
LIGHT            4
RDSFCOND        29
IMPACTYPE       27
INVTYPE         16
INJURY        8890
INITDIR       5272
VEHTYPE       3479
MANOEUVER     7945
DRIVACT       9281
DRIVCOND      9283
AUTOMOBILE    1723
AG_DRIV       9107
dtype: int64

✅ Missing values filled with "Unknown"

Missing values after cleaning:
0 total missing values remaining


## 2.6 Feature Engineering

### 2.6.1 Date-Based Features
### 2.6.2 Time-Based Features

Instead of using raw date and time values only, we engineer more meaningful features such as weekend, season, day/night, and rush hour.

These features are easier to interpret and more useful for machine learning.

In [6]:
# ============================================================
# Advanced Date & Time Feature Engineering
# ============================================================
# Convert DATE safely

df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

# Basic date features
df['YEAR'] = df['DATE'].dt.year
df['MONTH'] = df['DATE'].dt.month
df['MONTH_NAME'] = df['DATE'].dt.month_name()
df['DAY'] = df['DATE'].dt.day
df['DAY_OF_WEEK'] = df['DATE'].dt.day_name()

# WEEKEND
df['WEEKEND'] = df['DAY_OF_WEEK'].apply(
    lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday'
)

# SEASON
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['SEASON'] = df['MONTH'].apply(get_season)

# MONTH_PART
def get_month_part(day):
    if pd.isna(day):
        return 'Unknown'
    elif day <= 10:
        return 'Beginning'
    elif day <= 20:
        return 'Middle'
    else:
        return 'End'

df['MONTH_PART'] = df['DAY'].apply(get_month_part)

# Clean TIME and extract HOUR
df['TIME'] = df['TIME'].astype(str).str.zfill(4)
df['HOUR'] = pd.to_numeric(df['TIME'].str[:2], errors='coerce')

# DAY_NIGHT
df['DAY_NIGHT'] = df['HOUR'].apply(
    lambda x: 'Day' if pd.notna(x) and 6 <= x < 18 else 'Night'
)

# TIME_PERIOD
def get_time_period(hour):
    if pd.isna(hour):
        return 'Unknown'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['TIME_PERIOD'] = df['HOUR'].apply(get_time_period)

# RUSH_HOUR
df['RUSH_HOUR'] = df['HOUR'].apply(
    lambda x: 'Yes' if pd.notna(x) and ((7 <= x < 10) or (16 <= x < 19)) else 'No'
)

# Drop original DATE and TIME after feature extraction
df.drop(columns=['DATE', 'TIME'], inplace=True)

print('✅ Advanced date/time features extracted successfully')
print('   • YEAR, MONTH, MONTH_NAME, DAY, DAY_OF_WEEK added')
print('   • WEEKEND, SEASON, MONTH_PART added')
print('   • HOUR, DAY_NIGHT, TIME_PERIOD, RUSH_HOUR added')
print('   • DATE and TIME columns dropped')
print(f'\n📊 New shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

print("\nPreview of engineered features:")
print(df[['YEAR', 'MONTH', 'DAY_OF_WEEK', 'WEEKEND', 'SEASON', 'MONTH_PART', 'HOUR', 'DAY_NIGHT', 'TIME_PERIOD', 'RUSH_HOUR']].head())

✅ Advanced date/time features extracted successfully
   • YEAR, MONTH, MONTH_NAME, DAY, DAY_OF_WEEK added
   • WEEKEND, SEASON, MONTH_PART added
   • HOUR, DAY_NIGHT, TIME_PERIOD, RUSH_HOUR added
   • DATE and TIME columns dropped

📊 New shape: 18,938 rows × 34 columns

Preview of engineered features:
   YEAR  MONTH DAY_OF_WEEK  WEEKEND  SEASON MONTH_PART  HOUR DAY_NIGHT  \
0  2006      1      Sunday  Weekend  Winter  Beginning     2     Night   
1  2006      1      Sunday  Weekend  Winter  Beginning     2     Night   
2  2006      1      Sunday  Weekend  Winter  Beginning     2     Night   
3  2006      1      Sunday  Weekend  Winter  Beginning     2     Night   
4  2006      1      Sunday  Weekend  Winter  Beginning     2     Night   

  TIME_PERIOD RUSH_HOUR  
0       Night        No  
1       Night        No  
2       Night        No  
3       Night        No  
4       Night        No  


### 2.6.3 Geographic Features

Raw geographic coordinates are not always the best representation for machine learning, especially when location can be expressed more meaningfully through neighborhood and district information.

For this reason, we use neighborhood-level features to capture spatial patterns related to accident severity.

In [7]:
print("Geographic-related columns in the dataset:")
geo_cols = [col for col in df.columns if any(keyword in col.upper() for keyword in ['LAT', 'LONG', 'NEIGH', 'DISTRICT', 'WARD', 'ROAD'])]
print(geo_cols)

Geographic-related columns in the dataset:
['ROAD_CLASS', 'DISTRICT', 'NEIGHBOURHOOD_158']


In [8]:
# ============================================================
#           Geographic Feature Engineering
# ============================================================

# Make sure the neighbourhood column is treated as text
df['NEIGHBOURHOOD_158'] = df['NEIGHBOURHOOD_158'].astype(str)

# Count fatal collisions by neighbourhood
fatal_counts_by_neigh = (
    df[df['ACCLASS'] == 'Fatal']
    .groupby('NEIGHBOURHOOD_158')
    .size()
    .sort_values(ascending=False)
)

# Select top 10 neighbourhoods with the highest number of fatal collisions
top_high_risk_neigh = fatal_counts_by_neigh.head(10).index.tolist()

# Create a binary feature: 1 if the neighbourhood is high-risk, otherwise 0
df['HIGH_RISK_NEIGHBOURHOOD'] = df['NEIGHBOURHOOD_158'].apply(
    lambda x: 1 if x in top_high_risk_neigh else 0
)

print("✅ Geographic features prepared successfully")
print("\nTop 10 high-risk neighbourhoods based on fatal collisions:")
print(top_high_risk_neigh)

print("\nPreview:")
print(df[['DISTRICT', 'NEIGHBOURHOOD_158', 'HIGH_RISK_NEIGHBOURHOOD']].head())

✅ Geographic features prepared successfully

Top 10 high-risk neighbourhoods based on fatal collisions:
['West Humber-Clairville', 'Wexford/Maryvale', 'South Parkdale', 'Victoria Village', 'Clairlea-Birchmount', 'Steeles', 'Dorset Park', 'St Lawrence-East Bayfront-The Islands', 'Milliken', 'Kensington-Chinatown']

Preview:
                DISTRICT NEIGHBOURHOOD_158  HIGH_RISK_NEIGHBOURHOOD
0  Toronto and East York  Woodbine-Lumsden                        0
1  Toronto and East York  Woodbine-Lumsden                        0
2  Toronto and East York  Woodbine-Lumsden                        0
3  Toronto and East York  Woodbine-Lumsden                        0
4  Toronto and East York  Woodbine-Lumsden                        0


### Why these geographic features matter

Location is a key factor in traffic collision analysis.  
Instead of relying on raw coordinate values, we use neighborhood-level information to capture spatial risk patterns.

The new feature `HIGH_RISK_NEIGHBOURHOOD` highlights areas with historically higher numbers of fatal collisions.  
This may help the model detect geographic risk more effectively.

## 2.7 Handle Rare Categories

Some categorical variables may contain categories with very low frequency.  
Keeping all of them can introduce noise and create too many encoded features.

To reduce dimensionality and improve model generalization, rare categories are grouped into `Other`.

In [9]:
# ============================================================
# 2.7 — Handle Rare Categories
# ============================================================

# Columns to check for rare categories
rare_cat_cols = ['ROAD_CLASS', 'DISTRICT', 'NEIGHBOURHOOD_158', 'TIME_PERIOD', 'SEASON']

# Threshold: categories representing less than 1% of the dataset
threshold = 0.01

for col in rare_cat_cols:
    if col in df.columns:
        freq = df[col].value_counts(normalize=True)
        rare_values = freq[freq < threshold].index
        df[col] = df[col].replace(rare_values, 'Other')

print("✅ Rare categories handled successfully")

for col in rare_cat_cols:
    if col in df.columns:
        print(f"\n{col} - updated category distribution:")
        print(df[col].value_counts().head(10))

✅ Rare categories handled successfully

ROAD_CLASS - updated category distribution:
ROAD_CLASS
Major Arterial    13358
Minor Arterial     2957
Collector          1032
Local               865
Unknown             486
Other               240
Name: count, dtype: int64

DISTRICT - updated category distribution:
DISTRICT
Toronto and East York    6325
Etobicoke York           4338
Scarborough              4264
North York               3782
Unknown                   229
Name: count, dtype: int64

NEIGHBOURHOOD_158 - updated category distribution:
NEIGHBOURHOOD_158
Other                                    13077
West Humber-Clairville                     597
Yonge-Bay Corridor                         376
Wexford/Maryvale                           361
South Riverdale                            353
South Parkdale                             304
St Lawrence-East Bayfront-The Islands      303
Clairlea-Birchmount                        284
Kensington-Chinatown                       270
Moss Park     

### Why rare category grouping matters

Rare categories can create unnecessary complexity during encoding and may not provide enough information for the model to learn stable patterns.

By grouping infrequent values into `Other`, we reduce noise, simplify the feature space, and improve the robustness of the preprocessing pipeline.

## 2.8 Target Variable Encoding

In this section, we convert the original accident class into a binary target variable for classification.

- `1` = Fatal
- `0` = Non-Fatal Injury

This makes the target easier to use for supervised machine learning models.

In [10]:
# ============================================================
# 2.8 — Encode Target Variable
# ============================================================
# Fatal = 1, Non-Fatal Injury = 0
df['TARGET'] = (df['ACCLASS'] == 'Fatal').astype(int)

print('✅ Target variable encoded:')
print('   • Fatal         → 1')
print('   • Non-Fatal     → 0')
print(f'\nTarget distribution:')
print(df['TARGET'].value_counts())
print(f'\nPercentage:')
print(df['TARGET'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

✅ Target variable encoded:
   • Fatal         → 1
   • Non-Fatal     → 0

Target distribution:
TARGET
0    16268
1     2670
Name: count, dtype: int64

Percentage:
TARGET
0    85.9%
1    14.1%
Name: proportion, dtype: object


## 2.9 Encode Categorical Variables

Machine learning models require numerical input.  
In this section, categorical variables are converted into numeric form so they can be used by classification algorithms.

This step is necessary to prepare the dataset for model training.

In [11]:
# ============================================================
# 2.9 — Encode Categorical Variables (Label Encoding)
# ============================================================
from sklearn.preprocessing import LabelEncoder

# Drop ACCLASS (replaced by TARGET)
df.drop(columns=['ACCLASS'], inplace=True)

# Identify categorical columns to encode
cat_cols_to_encode = df.select_dtypes(include='object').columns.tolist()

print(f'📊 Categorical columns to encode: {len(cat_cols_to_encode)}')
print('='*45)

le = LabelEncoder()
for col in cat_cols_to_encode:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f'   ✅ {col} encoded')

print(f'\n📊 Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'✅ All categorical columns encoded successfully')

📊 Categorical columns to encode: 29
   ✅ ROAD_CLASS encoded
   ✅ DISTRICT encoded
   ✅ ACCLOC encoded
   ✅ TRAFFCTL encoded
   ✅ VISIBILITY encoded
   ✅ LIGHT encoded
   ✅ RDSFCOND encoded
   ✅ IMPACTYPE encoded
   ✅ INVTYPE encoded
   ✅ INVAGE encoded
   ✅ INJURY encoded
   ✅ INITDIR encoded
   ✅ VEHTYPE encoded
   ✅ MANOEUVER encoded
   ✅ DRIVACT encoded
   ✅ DRIVCOND encoded
   ✅ AUTOMOBILE encoded
   ✅ AG_DRIV encoded
   ✅ HOOD_158 encoded
   ✅ NEIGHBOURHOOD_158 encoded
   ✅ DIVISION encoded
   ✅ MONTH_NAME encoded
   ✅ DAY_OF_WEEK encoded
   ✅ WEEKEND encoded
   ✅ SEASON encoded
   ✅ MONTH_PART encoded
   ✅ DAY_NIGHT encoded
   ✅ TIME_PERIOD encoded
   ✅ RUSH_HOUR encoded

📊 Final shape: 18,938 rows × 35 columns
✅ All categorical columns encoded successfully


## 2.10 Scale Numerical Features

Numerical variables may have different ranges and magnitudes.  
In this section, we standardize the numerical features so they are on a comparable scale.

This step is especially useful for machine learning models that are sensitive to feature magnitude.

In [12]:
# ============================================================
# 2.10 — Normalize Numerical Features
# ============================================================
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('✅ Features normalized with StandardScaler')
print(f'📊 Features shape: {X_scaled.shape[0]:,} rows × {X_scaled.shape[1]} columns')
print(f'📊 Target shape: {y.shape[0]:,} rows')
print(f'\nSample of normalized data:')
X_scaled.head()

✅ Features normalized with StandardScaler
📊 Features shape: 18,938 rows × 34 columns
📊 Target shape: 18,938 rows

Sample of normalized data:


,ROAD_CLASS,DISTRICT,ACCLOC,TRAFFCTL,VISIBILITY,LIGHT,RDSFCOND,IMPACTYPE,INVTYPE,INVAGE,INJURY,INITDIR,VEHTYPE,MANOEUVER,DRIVACT,DRIVCOND,AUTOMOBILE,AG_DRIV,HOOD_158,NEIGHBOURHOOD_158,DIVISION,YEAR,MONTH,MONTH_NAME,DAY,DAY_OF_WEEK,WEEKEND,SEASON,MONTH_PART,HOUR,DAY_NIGHT,TIME_PERIOD,RUSH_HOUR,HIGH_RISK_NEIGHBOURHOOD
0,-0.124915,1.095091,-0.386001,-0.994167,-0.390866,-1.46889,2.191626,-1.384684,0.727226,0.122185,-1.068241,0.633621,1.743427,0.902017,0.913363,0.741515,0.316366,0.962474,0.939388,-0.033667,1.597885,-1.451788,-1.787034,-0.490588,-1.648525,0.019323,1.618747,1.425596,-1.22061,-1.799402,1.117065,1.322546,-0.65495,-0.422803
1,-0.124915,1.095091,-0.386001,-0.994167,-0.390866,-1.46889,2.191626,-1.384684,0.727226,-1.258445,0.288908,0.633621,1.743427,0.902017,0.913363,0.741515,0.316366,0.962474,0.939388,-0.033667,1.597885,-1.451788,-1.787034,-0.490588,-1.648525,0.019323,1.618747,1.425596,-1.22061,-1.799402,1.117065,1.322546,-0.65495,-0.422803
2,-0.124915,1.095091,-0.386001,-0.994167,-0.390866,-1.46889,2.191626,-1.384684,-0.900908,0.294764,0.288908,-0.826263,-0.911435,-1.264489,-1.342383,-0.489285,0.316366,0.962474,0.939388,-0.033667,1.597885,-1.451788,-1.787034,-0.490588,-1.648525,0.019323,1.618747,1.425596,-1.22061,-1.799402,1.117065,1.322546,-0.65495,-0.422803
3,-0.124915,1.095091,-0.386001,-0.994167,-0.390866,-1.46889,2.191626,-1.384684,0.727226,-1.085866,0.288908,0.633621,1.743427,0.902017,0.913363,0.741515,0.316366,0.962474,0.939388,-0.033667,1.597885,-1.451788,-1.787034,-0.490588,-1.648525,0.019323,1.618747,1.425596,-1.22061,-1.799402,1.117065,1.322546,-0.65495,-0.422803
4,-0.124915,1.095091,-0.386001,-0.994167,-0.390866,-1.46889,2.191626,-1.384684,0.727226,-1.258445,0.288908,0.633621,1.743427,0.902017,0.913363,0.741515,0.316366,0.962474,0.939388,-0.033667,1.597885,-1.451788,-1.787034,-0.490588,-1.648525,0.019323,1.618747,1.425596,-1.22061,-1.799402,1.117065,1.322546,-0.65495,-0.422803


## 2.11 Train/Test Split

To evaluate the machine learning models properly, the dataset is divided into training and testing sets.

The training set is used to learn patterns from the data, while the testing set is used to evaluate how well the model performs on unseen data.

In [13]:
# ============================================================
# 2.11 — Train/Test Split
# ============================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print('✅ Train/Test Split completed')
print(f'\n📊 Training set   : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X_scaled)*100:.1f}%)')
print(f'📊 Test set       : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X_scaled)*100:.1f}%)')
print(f'\nTarget distribution in training set:')
print(y_train.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
print(f'\nTarget distribution in test set:')
print(y_test.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

✅ Train/Test Split completed

📊 Training set   : 15,150 rows (80.0%)
📊 Test set       : 3,788 rows (20.0%)

Target distribution in training set:
TARGET
0    85.9%
1    14.1%
Name: proportion, dtype: object

Target distribution in test set:
TARGET
0    85.9%
1    14.1%
Name: proportion, dtype: object


## 2.12 Handling Imbalanced Data (SMOTE)

The dataset is highly imbalanced, with far fewer fatal collisions compared to non-fatal ones.

To address this issue, we use SMOTE (Synthetic Minority Over-sampling Technique).

SMOTE creates synthetic examples of the minority class, helping balance the training data and improve the model’s ability to detect fatal accidents.

In [14]:
# ============================================================
# 2.12 — Handle Imbalanced Classes (SMOTE)
# ============================================================
from imblearn.over_sampling import SMOTE

print('Before SMOTE:')
print(f'   Fatal (1)     : {sum(y_train == 1):,} ({sum(y_train == 1)/len(y_train)*100:.1f}%)')
print(f'   Non-Fatal (0) : {sum(y_train == 0):,} ({sum(y_train == 0)/len(y_train)*100:.1f}%)')

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE:')
print(f'   Fatal (1)     : {sum(y_train_smote == 1):,} ({sum(y_train_smote == 1)/len(y_train_smote)*100:.1f}%)')
print(f'   Non-Fatal (0) : {sum(y_train_smote == 0):,} ({sum(y_train_smote == 0)/len(y_train_smote)*100:.1f}%)')
print(f'\n✅ Classes are now balanced!')
print(f'📊 New training set size: {len(X_train_smote):,} rows')

Before SMOTE:
   Fatal (1)     : 2,136 (14.1%)
   Non-Fatal (0) : 13,014 (85.9%)

After SMOTE:
   Fatal (1)     : 13,014 (50.0%)
   Non-Fatal (0) : 13,014 (50.0%)

✅ Classes are now balanced!
📊 New training set size: 26,028 rows


## 2.13 Pipeline

We use a machine learning pipeline to combine preprocessing and model training into a single workflow.

This helps simplify the process and ensures that all transformations are applied consistently.

In [15]:
# ============================================================
# 2.13 — Pipeline Preparation
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

print("Pipeline created successfully")

Pipeline created successfully


## 2.14 Save Prepared Files

In this section, we save the processed datasets and export the cleaned file for reuse in modeling and Power BI visualization.

In [19]:
# ============================================================
# 2.14 — Save Cleaned Data
# ============================================================
import pickle
import os

MODELS_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\models\\'

os.makedirs(MODELS_PATH, exist_ok=True)

# Save train/test sets
pickle.dump(X_train_smote, open(MODELS_PATH + 'X_train.pkl', 'wb'))
pickle.dump(X_test,        open(MODELS_PATH + 'X_test.pkl', 'wb'))
pickle.dump(y_train_smote, open(MODELS_PATH + 'y_train.pkl', 'wb'))
pickle.dump(y_test,        open(MODELS_PATH + 'y_test.pkl', 'wb'))

# Save cleaned dataframe for Power BI
POWERBI_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\powerbi\\'

df_powerbi = df.copy()
df_powerbi.to_csv(POWERBI_PATH + 'KSI_PowerBI_Clean.csv', index=False)

print('✅ All data saved successfully!')
print(f'\n📁 Models folder:')
print(f'   • X_train.pkl — {len(X_train_smote):,} rows')
print(f'   • X_test.pkl  — {len(X_test):,} rows')
print(f'   • y_train.pkl — {len(y_train_smote):,} rows')
print(f'   • y_test.pkl  — {len(y_test):,} rows')
print(f'\n📁 Power BI folder:')
print(f'   • KSI_PowerBI_Clean.csv — {len(df_powerbi):,} rows')

✅ All data saved successfully!

📁 Models folder:
   • X_train.pkl — 26,028 rows
   • X_test.pkl  — 3,788 rows
   • y_train.pkl — 26,028 rows
   • y_test.pkl  — 3,788 rows

📁 Power BI folder:
   • KSI_PowerBI_Clean.csv — 18,938 rows


In [17]:
# ============================================================
# 2.15 — Step 2 Summary
# ============================================================
print('=' * 60)
print('✅ STEP 2 — DATA CLEANING, PREPROCESSING & FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'Original dataset             : 18,957 rows × 54 columns')
print(f'After removing PD Only       : 18,938 rows × 54 columns')
print(f'After dropping >50% NaN      : 18,938 rows × 35 columns')
print(f'After dropping weak columns  : 18,938 rows × 24 columns')
print(f'After feature engineering    : 18,938 rows × 35 columns')
print(f'Final processed dataset      : {df.shape[0]:,} rows × {df.shape[1]} columns')
print('=' * 60)
print(f'X_train shape                : {X_train.shape}')
print(f'X_test shape                 : {X_test.shape}')
print(f'y_train shape                : {y_train.shape}')
print(f'y_test shape                 : {y_test.shape}')
print('=' * 60)
print(f'Training set after SMOTE     : 26,028 rows (balanced)')
print(f'Test set                     : 3,788 rows (original distribution preserved)')
print('=' * 60)
print('\n📌 New engineered features added:')
print('   • WEEKEND')
print('   • SEASON')
print('   • MONTH_PART')
print('   • DAY_NIGHT')
print('   • TIME_PERIOD')
print('   • RUSH_HOUR')
print('   • HIGH_RISK_NEIGHBOURHOOD')
print('\n📁 Files saved:')
print('   • models/X_train.pkl')
print('   • models/X_test.pkl')
print('   • models/y_train.pkl')
print('   • models/y_test.pkl')
print('   • powerbi/KSI_PowerBI_Clean.csv')
print('\n➡️ Next Step: Step 3 — Model Building and Evaluation')

✅ STEP 2 — DATA CLEANING, PREPROCESSING & FEATURE ENGINEERING SUMMARY
Original dataset             : 18,957 rows × 54 columns
After removing PD Only       : 18,938 rows × 54 columns
After dropping >50% NaN      : 18,938 rows × 35 columns
After dropping weak columns  : 18,938 rows × 24 columns
After feature engineering    : 18,938 rows × 35 columns
Final processed dataset      : 18,938 rows × 35 columns
X_train shape                : (15150, 34)
X_test shape                 : (3788, 34)
y_train shape                : (15150,)
y_test shape                 : (3788,)
Training set after SMOTE     : 26,028 rows (balanced)
Test set                     : 3,788 rows (original distribution preserved)

📌 New engineered features added:
   • WEEKEND
   • SEASON
   • MONTH_PART
   • DAY_NIGHT
   • TIME_PERIOD
   • RUSH_HOUR
   • HIGH_RISK_NEIGHBOURHOOD

📁 Files saved:
   • models/X_train.pkl
   • models/X_test.pkl
   • models/y_train.pkl
   • models/y_test.pkl
   • powerbi/KSI_PowerBI_Clean.csv

➡️ 

## 2.16 Future Improvements

In the next phase of the project, we plan to:
- compare multiple classification models such as Logistic Regression, Decision Tree, Random Forest, SVM, and Neural Network
- perform hyperparameter tuning to improve model performance
- evaluate feature importance and the contribution of engineered variables
- compare model performance using accuracy, precision, recall, F1-score, confusion matrix, and ROC curve
- deploy the best model as a local API using Flask
- create a simple interface or test the API using Postman

These next steps will help transform the prepared dataset into a complete end-to-end machine learning solution.